In [1]:
import os
import json
import fitz  # PyMuPDF
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

DOCS_DIR = "documents"
INDEX_PATH = "faiss_index.bin"
CHUNKS_PATH = "chunks.json"
CHUNK_SIZE = 500  # words per chunk

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

def split_into_chunks(text, chunk_size=CHUNK_SIZE):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

def main():
    print("Loading embedding model...")
    model = SentenceTransformer("all-MiniLM-L6-v2")

    all_chunks = []

    for filename in os.listdir(DOCS_DIR):
        if filename.endswith(".pdf"):
            path = os.path.join(DOCS_DIR, filename)
            print(f"Processing: {filename}")
            text = extract_text_from_pdf(path)
            chunks = split_into_chunks(text)
            for chunk in chunks:
                all_chunks.append({"source": filename, "text": chunk})

    print(f"Total chunks: {len(all_chunks)}")

    texts = [c["text"] for c in all_chunks]
    print("Embedding chunks...")
    embeddings = model.encode(texts, show_progress_bar=True)
    embeddings = np.array(embeddings).astype("float32")

    # Build FAISS index
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)

    faiss.write_index(index, INDEX_PATH)
    with open(CHUNKS_PATH, "w") as f:
        json.dump(all_chunks, f)

    print(f"Done! Saved index to {INDEX_PATH}")

if __name__ == "__main__":
    main()

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Tanmay\anaconda3\envs\ragmcp\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tanmay\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Processing: 1706.03762v7.pdf
Processing: 1810.04805v2.pdf
Processing: 2005.11401v4.pdf
Total chunks: 54
Embedding chunks...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Done! Saved index to faiss_index.bin
